## AI4Climate ML tutorial - Training in PyTorch
* Author: Stephen Haddad
* Affiliation: UK Met Office
* History: 1.0
* Last update: 2026-02-10
* © British Crown Copyright 2017-2065, Met Office. Please see LICENSE.md for license details.

## Overview of the broad topic covered

In this notebook, we will use the previously prepared tabular dataset to predict climate zones. We will train on different eras to see how well the results generalise with climate change. 

### Prerequisites 
what background information is needed to go through the notebook
- Same as previouis notebooks
- Have completed training pipeline, inference and evaluation notebooks.

### Learning outcomes from completing the notebook

- Understand how to build a pipeline using pytorch
- Understand the core pytorch concepts and classes
- Understand bho to manage experiments using ML Flow

## Tutorial - Creating a machine learning pipe

Further Reading
* [PyTorch Docs](https://pytorch.org/)

## Setup
First we start by loading the data we have prepared previously, and other set up elements

### Imports

In [1]:
import pathlib
import os
import datetime
import json

In [2]:
import numpy 
import pandas

In [3]:
import matplotlib
import matplotlib.pyplot

In [4]:
import sklearn
import sklearn.preprocessing
import sklearn.tree


In [5]:
import mlflow

In [6]:
import torch

## Load and prepare data 
We will now load the dataset and do the usual data prep steps, like train/test split and normalisation.


#### Dataset parameters

In [7]:
with open ('config.json','r') as tutorial_config:
    tutorial_config = json.load(tutorial_config)
tutorial_config

{'platform': 'jasmin',
 'default_dirs': {'mo_linux': '/data/users/dscop/ml_tutorial/',
  'jasmin': '/gws/nopw/j04/mohc_shared/dscop/'},
 'random_seed': 12345,
 'climate_subgroups': {'non-land': 'None',
  'Af': 'Tropical, rainforest',
  'Am': 'Tropical, monsoon',
  'Aw': 'Tropical, savannah',
  'BWh': 'Arid, desert, hot',
  'BWk': 'Arid, desert, cold',
  'BSh': 'Arid, steppe, hot',
  'BSk': 'Arid, steppe, cold',
  'Csa': 'Temperate, dry summer, hot summer',
  'Csb': 'Temperate, dry summer, warm summer',
  'Csc': 'Temperate, dry summer, cold summer',
  'Cwa': 'Temperate, dry winter, hot summer',
  'Cwb': 'Temperate, dry winter, warm summer',
  'Cwc': 'Temperate, dry winter, cold summer',
  'Cfa': 'Temperate, no dry season, hot summer',
  'Cfb': 'Temperate, no dry season, warm summer',
  'Cfc': 'Temperate, no dry season, cold summer',
  'Dsa': 'Cold, dry summer, hot summer',
  'Dsb': 'Cold, dry summer, warm summer',
  'Dsc': 'Cold, dry summer, cold summer',
  'Dsd': 'Cold, dry summer, ver

In [8]:
def get_platform_dir(select_platform, config):
    try:
        root_path = pathlib.Path(config['default_dirs'][select_platform]) / 'climate_zones'
    except KeyError:
        root_path = pathlib.Path(os.environ['HOME']) / 'climate_zones'
    return root_path

In [9]:
current_platform = tutorial_config['platform']

In [10]:
current_platform

'jasmin'

In [11]:
root_data_dir = get_platform_dir(current_platform, tutorial_config)

print(root_data_dir.is_dir())
root_data_dir

True


PosixPath('/gws/nopw/j04/mohc_shared/dscop/climate_zones')

In [12]:
ml_ready_dir = root_data_dir / 'ml_ready'
print(ml_ready_dir.is_dir())
ml_ready_dir

True


PosixPath('/gws/nopw/j04/mohc_shared/dscop/climate_zones/ml_ready')

In [13]:
resolutions_dict = {float(k1): v1 for k1,v1 in tutorial_config['resolutions_names'].items()}

dataset_prefix_dict = tutorial_config['dataset_prefix']

format_str = 'nc'
historic_scenario_str = 'historic'

future_scenario_list = tutorial_config['future_scenarios']
historic_scenario_list = tutorial_config['historic_scenarios']

time_periods = { 
    (1901,1930): historic_scenario_list, 
    (1931,1960): historic_scenario_list,
    (1961,1990): historic_scenario_list,
    (1991,2020): historic_scenario_list,
    (2041,2070): future_scenario_list,
    (2071,2099): future_scenario_list,
}


In [14]:
fname_template = tutorial_config['fname_template']
time_dir_template = tutorial_config['time_dir_template']
ml_ready_fname_template = tutorial_config['csv_out_template']

#### Load data for training

In [15]:
current_res = 1.0

In [16]:
mlready_data_path = ml_ready_dir / ml_ready_fname_template.format(resolution=resolutions_dict[current_res])
print(mlready_data_path.is_file())
mlready_data_path

True


PosixPath('/gws/nopw/j04/mohc_shared/dscop/climate_zones/ml_ready/climate_zones_1p0.csv')

In [17]:
zones_df = pandas.read_csv(mlready_data_path)

We have two options for the target. We have the full 30 class climate subgroups of the Koppen-Geiger classification data from the original dataset. We also have the processed five class climate group target, which presents an easier target for our classification algorithm to predict.

#### Selecting features

In [18]:
predictors_dict = {
    'precip_mean': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'mean' in c1],
    'precip_std': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'std' in c1],
    'temp_mean': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'mean' in c1],
    'temp_std': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'std' in c1],
}


In [19]:
target_var = 'climate_group' # 5 classes
# target_var = 'climate_subgroup # 30 classes


In [20]:
predictors = predictors_dict['precip_mean'] + predictors_dict['temp_mean']
predictors

['precipitation_1.0_mean',
 'precipitation_2.0_mean',
 'precipitation_3.0_mean',
 'precipitation_4.0_mean',
 'precipitation_5.0_mean',
 'precipitation_6.0_mean',
 'precipitation_7.0_mean',
 'precipitation_8.0_mean',
 'precipitation_9.0_mean',
 'precipitation_10.0_mean',
 'precipitation_11.0_mean',
 'precipitation_12.0_mean',
 'air_temperature_1.0_mean',
 'air_temperature_2.0_mean',
 'air_temperature_3.0_mean',
 'air_temperature_4.0_mean',
 'air_temperature_5.0_mean',
 'air_temperature_6.0_mean',
 'air_temperature_7.0_mean',
 'air_temperature_8.0_mean',
 'air_temperature_9.0_mean',
 'air_temperature_10.0_mean',
 'air_temperature_11.0_mean',
 'air_temperature_12.0_mean']

#### Train/test split

In [21]:
random_seed = tutorial_config['random_seed']

In [22]:
test_frac = 0.1
val_frac = 0.1
val_frac_sub = (val_frac / (1.0-test_frac) )

In [23]:
test_df = zones_df.groupby(['period_start','scenario']).sample(frac=test_frac, random_state=random_seed)
remain_df = zones_df.drop(test_df.index)

In [24]:
val_df = remain_df.groupby(['period_start','scenario']).sample(frac=val_frac_sub, random_state=random_seed)
train_df = remain_df.drop(val_df.index)


## Using Pytorch

From this point in the tutorial, we diverge from what was done in the ml training pipeline tutorial as instead of setting up and training our model in scikit-learn, we're going to use a more sophisticated machine learning library called pytorch. This gives us more more control over how we implement our neural network and gives us much more power to use advanced architectures, losss functions and distributed computing techniques.

### Check GPU availability

In [25]:
# Autodetect GPU and use if possible
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

#### Training hyperparameters

In [26]:
training_params = {
    'batch_size': 16,
    'num_epochs': 10,
    'learning_rate': 0.001,
    'loss': 'CrossEntropyLoss',
    'criterion': 'CrossEntropyLoss',
    'optimizer': 'Adam',
}

### Define a data loader

Further reading
- https://docs.pytorch.org/tutorials/beginner/data_loading_tutorial.html
- https://machinelearningmastery.com/converting-pandas-dataframes-to-pytorch-dataloaders-for-custom-deep-learning-model-training/
- https://www.geeksforgeeks.org/deep-learning/converting-a-pandas-dataframe-to-a-pytorch-tensor/
- https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelBinarizer.html

In [27]:
class ClimateZonesDataset(torch.utils.data.Dataset):
    """
    Inspired by this tutorial:
    https://machinelearningmastery.com/converting-pandas-dataframes-to-pytorch-dataloaders-for-custom-deep-learning-model-training/
    """
    def __init__(self, df_ml, predictor_features, target_feature, device, stats_dict=None):
        self._df_ml = df_ml.reset_index().drop(['index'],axis='columns')
        self._device = device
        self.input_scaler = sklearn.preprocessing.StandardScaler()
        if stats_dict is None:
            self.input_scaler.fit(self._df_ml[predictor_features])
        else:
            self.input_scaler.mean_ = numpy.array(stats_dict['input_mean'])
            self.input_scaler.scale_ = numpy.array(stats_dict['input_scale'])

        self._X = torch.tensor(self.input_scaler.transform(self._df_ml[predictor_features]),  
                               dtype=torch.float32)


        self.target_encoder = sklearn.preprocessing.LabelBinarizer(sparse_output=False)
        if stats_dict is None:
            self.target_encoder.fit(self._df_ml[[target_feature]])
        else:
            self.target_encoder.classes_ = numpy.array(stats_dict['target_classes'],dtype='object')

        
        self._y = torch.tensor(self.target_encoder.transform(self._df_ml[[target_feature]]),
                               dtype=torch.float32)

        self.stats_dict = {
            'input_mean': [float(v1) for v1 in self.input_scaler.mean_],
            'input_scale': [float(v1) for v1 in self.input_scaler.scale_],
            'target_classes': list(self.target_encoder.classes_),
        }
        
    def _repr_html_(self):
        return f'''
        <h1>Climate Zones Dataset</h1>
        Number of samples {len(self._X)}
        '''
    
    def __len__(self):
        return len(self._X)

    def __getitem__(self,idx):
        return self._X[idx], self._y[idx]


        

In [28]:
cz_train_ds = ClimateZonesDataset(train_df, predictors, target_var, device)
cz_train_ds

We now intialise the validate and test set data loaders. Note that we initialise the preprocessing objects with the values learned from the training data, rather than calculating them on the validate or test data.

In [29]:
cz_val_ds = ClimateZonesDataset(val_df, predictors, target_var, device, cz_train_ds.stats_dict)
cz_test_ds = ClimateZonesDataset(test_df, predictors, target_var, device, cz_train_ds.stats_dict)

/home/users/shaddad/mohc_user/venv/ai4c_nb_gpu/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/users/shaddad/mohc_user/venv/ai4c_nb_gpu/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [30]:
cz_val_ds[10:13]

(tensor([[-0.4408, -0.3281, -0.1737, -0.2187, -0.2642, -0.3904, -0.3665, -0.3730,
          -0.4555, -0.4177, -0.4437, -0.4373, -0.5886, -0.7146, -0.8674, -0.9889,
          -1.0984, -1.1859, -1.2289, -1.2352, -1.1710, -1.0693, -0.9048, -0.6668],
         [-0.5651, -0.5227, -0.5370, -0.5412, -0.5645, -0.5760, -0.5667, -0.5496,
          -0.6736, -0.6741, -0.6055, -0.5566, -0.9222, -1.0107, -1.1763, -1.3347,
          -1.4611, -1.4826, -1.4871, -1.4774, -1.4484, -1.4140, -1.2316, -0.9931],
         [-0.5389, -0.5087, -0.4206, -0.4052, -0.3257, -0.5483, -0.5183, -0.5140,
          -0.5723, -0.5571, -0.6006, -0.5566, -0.2212, -0.4738, -0.7350, -0.8927,
          -1.0188, -1.0741, -1.1504, -1.2124, -1.1016, -0.9153, -0.5810, -0.2653]]),
 tensor([[0., 0., 0., 0., 1.],
         [0., 0., 0., 0., 1.],
         [0., 0., 0., 0., 1.]]))

In [32]:
cz_train_loader = torch.utils.data.DataLoader(
        cz_train_ds, batch_size=training_params['batch_size'], shuffle=True, num_workers=1,
    )
cz_val_loader = torch.utils.data.DataLoader(
        cz_val_ds, batch_size=training_params['batch_size'], shuffle=False, num_workers=1,
    )

In [33]:
count = 0
for i1 in cz_train_loader:
    print(i1)
    count +=1
    if count > 5:
        break

[tensor([[-0.6107, -0.6325, -0.6667, -0.7035, -0.7314, -0.7021, -0.7207, -0.7727,
         -0.7768, -0.7666, -0.6967, -0.6405,  0.8348,  0.9239,  0.9835,  1.0245,
          1.0399,  1.0026,  0.9646,  0.9900,  1.0118,  1.0334,  0.9834,  0.8704],
        [-0.3403, -0.3989, -0.4890, -0.5558, -0.5328, -0.1910, -0.0086, -0.1189,
         -0.2175, -0.2657, -0.2476, -0.3295, -1.3232, -1.0606, -0.6100, -0.2009,
          0.1664,  0.4049,  0.4255,  0.3235,  0.1838, -0.1916, -0.7892, -1.2911],
        [-0.4366, -0.5003, -0.5157, -0.4295, -0.2549, -0.0282, -0.0274, -0.0306,
         -0.2329, -0.3505, -0.3437, -0.3799, -1.5161, -1.1408, -0.5291, -0.1757,
          0.1001,  0.3383,  0.4359,  0.3297,  0.1188, -0.2967, -1.0552, -1.4322],
        [-0.3546, -0.1587,  0.0785,  0.0979,  0.0445, -0.1650, -0.1373, -0.1863,
         -0.1089, -0.1185, -0.2545, -0.2747, -0.4233, -0.5707, -0.7105, -0.8766,
         -0.9790, -1.0741, -1.1215, -1.1586, -1.0843, -0.9642, -0.7718, -0.5099],
        [-0.6014, -0.60

## Building a pytorch model
- https://machinelearningmastery.com/building-a-multiclass-classification-model-in-pytorch/ 

In [49]:
class ClimateZoneClassifier(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = torch.nn.Linear(24, 60)
        self.act1 = torch.nn.ReLU()
        self.layer2 = torch.nn.Linear(60, 60)
        self.act2 = torch.nn.ReLU()
        self.layer3 = torch.nn.Linear(60, 60)
        self.act3 = torch.nn.ReLU()
        self.output = torch.nn.Linear(60, 5)
        self.sigmoid = torch.nn.Sigmoid()
        self.lsm = torch.nn.Softmax(dim=-1)
 
    def forward(self, x):
        x = self.act1(self.layer1(x))
        x = self.act2(self.layer2(x))
        x = self.act3(self.layer3(x))
        # x = self.sigmoid(self.output(x))
        x = self.lsm(self.output(x))
        return x
    


In [50]:
cz_classifier = ClimateZoneClassifier().to(device)

In [51]:
cz_classifier(cz_train_ds[:10][0].to(device))

tensor([[0.1941, 0.2004, 0.2083, 0.2000, 0.1971],
        [0.1942, 0.2000, 0.2086, 0.2002, 0.1971],
        [0.1942, 0.1996, 0.2088, 0.2003, 0.1971],
        [0.1943, 0.1995, 0.2089, 0.2003, 0.1970],
        [0.1944, 0.1992, 0.2090, 0.2004, 0.1970],
        [0.1943, 0.1991, 0.2090, 0.2006, 0.1971],
        [0.1941, 0.1989, 0.2089, 0.2007, 0.1974],
        [0.1939, 0.1989, 0.2088, 0.2008, 0.1976],
        [0.1939, 0.1989, 0.2088, 0.2008, 0.1976],
        [0.1937, 0.1988, 0.2088, 0.2010, 0.1978]], device='cuda:0',
       grad_fn=<SoftmaxBackward0>)

In [52]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cz_classifier.parameters(), 
                             lr=training_params['learning_rate'])

### Experiment tracking with mlflow
In a project to develop a machine learning model, we are likely to train many different models as part of our epxerimentation, with different prdictors, hyperparameters, architectures and other experimental choices that we vary to understand the problem and find the best solution. We will ave a collection of different models, and to properly asses our experiments we need to know exactly which set of choices go with which model. Experiment tracking tools log all the elements of a training run together so they can be retrieved and analysed later. A common tool for this is **ML Flow**.

Further Reading
- [ML Flow docs]()
- [Tracking pytorch with mlflow](https://mlflow.org/docs/latest/ml/deep-learning/pytorch/)


In [53]:
import mlflow

In [54]:
try:
    mlflow_port = os.environ['MLFLOW_PORT']
except KeyError:
    mlflow_port = 4455    
mlflow_server_uri = f'http://localhost:{mlflow_port}'


In [55]:
print(f'connecting to mlflow server {mlflow_server_uri}')
mlflow.set_tracking_uri(mlflow_server_uri)

connecting to mlflow server http://localhost:4455


In [56]:
mlflow.pytorch.autolog()

In [57]:
exp_name='climate_zones_torch_nn'

In [58]:
if mlflow.get_experiment_by_name(exp_name) is None:
    exp_id = mlflow.create_experiment(exp_name)
exp1 = mlflow.get_experiment_by_name(exp_name)
exp1

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1772795630343, experiment_id='2', last_update_time=1772795630343, lifecycle_stage='active', name='climate_zones_torch_nn', tags={}>

In [59]:
mlflow.models.infer_signature(cz_train_ds[:5][0].numpy(), cz_train_ds[:5][1].numpy())

inputs: 
  [Tensor('float32', (-1, 24))]
outputs: 
  [Tensor('float32', (-1, 5))]
params: 
  None

In [61]:
cz_signature = mlflow.models.infer_signature(cz_train_ds[:5][0].numpy(), cz_train_ds[:5][1].numpy())

In [62]:
def get_classification_metrics(model, train_ds, val_ds, class_labels):
    target_encoder = train_ds.target_encoder
    return pandas.DataFrame({
        'climate_group': class_labels,
        'precision_train': sklearn.metrics.precision_score(
            target_encoder.inverse_transform(model(train_ds._X.to(device)).to('cpu').detach().numpy()),
            target_encoder.inverse_transform(train_ds._y.numpy()),
            average=None,
            labels=class_labels,
        ),
        'recall_train': sklearn.metrics.recall_score(
            target_encoder.inverse_transform(model(train_ds._X.to(device)).to('cpu').detach().numpy()),
            target_encoder.inverse_transform(train_ds._y.numpy()),
            average=None,
            labels=class_labels,
        ),
        'precision_val': sklearn.metrics.precision_score(
            target_encoder.inverse_transform(model(val_ds._X.to(device)).to('cpu').detach().numpy()),
            target_encoder.inverse_transform(val_ds._y.numpy()),
            average=None,
            labels=class_labels,
        ),
        'recall_val': sklearn.metrics.recall_score(
            target_encoder.inverse_transform(model(val_ds._X.to(device)).to('cpu').detach().numpy()),
            target_encoder.inverse_transform(val_ds._y.numpy()),
            average=None,
            labels=class_labels,
        ),        
    })

### Run the training loop

In [69]:
%%time
with mlflow.start_run(experiment_id=exp1.experiment_id) as current_run:
    mlflow.log_params(training_params)    
    mlflow.log_dict(cz_train_ds.stats_dict, 'stats.json')
    for epoch in range(training_params['num_epochs']):
        print(f'epoch {epoch}')
        cz_classifier.train()
        epoch_loss_train = 0.0
        for batch_X, batch_y in cz_train_loader:
            optimizer.zero_grad()
            predictions = cz_classifier(batch_X.to(device))
            loss = loss_fn(predictions, batch_y.to(device))
            loss.backward()
            optimizer.step()
            epoch_loss_train += loss.to('cpu').item()

        #divide by number of batches
        epoch_loss_train /= len(cz_train_loader)
        
        epoch_loss_val = 0.0
        for X_val, y_val in cz_val_loader:
            epoch_loss_val += loss_fn(cz_classifier(X_val.to(device)), y_val.to(device)).item()    
        epoch_loss_val /= len(cz_val_loader)

        mlflow.log_metrics(
            { 'cross_entropy_train': epoch_loss_train,
            'cross_entropy_val': epoch_loss_val, },
            step=epoch,
        )
        
    mlflow.pytorch.log_model(cz_classifier,
                         name='climate_zones_classifier_torch', 
                         signature=cz_signature, 
                         input_example = cz_train_ds[:5][0].numpy(),
                         export_model=True,
                        )
    
    epoch_metrics = get_classification_metrics(cz_classifier, cz_train_ds, cz_val_ds, )
    mlflow.log_table(epoch_metrics, 'metrics.csv')
        



epoch 0
epoch 1
epoch 2
epoch 3
epoch 4
epoch 5
epoch 6
epoch 7
epoch 8
epoch 9
🏃 View run selective-pig-348 at: http://localhost:4455/#/experiments/2/runs/f1802f7630a54a479aa9f5669d9202f5
🧪 View experiment at: http://localhost:4455/#/experiments/2
CPU times: user 6min 34s, sys: 55.8 s, total: 7min 30s
Wall time: 8min 11s


NameError: name 'class_labels' is not defined

In [ ]:
cz_classifier(cz_train_ds[12345:12350][0].to(device))

In [ ]:
 loss_fn(cz_classifier(cz_train_ds[:100][0].to(device)), cz_train_ds[:100][1].to(device)).item()

In [ ]:
float(loss_fn( cz_train_ds[14350:14359][1].to(device), cz_train_ds[14350:14359][1].to(device)).to('cpu'))

In [ ]:
mlflow.pytorch.log_model

In [ ]:
(cz_train_ds.target_encoder.inverse_transform(cz_classifier(cz_train_ds[11230:11235][0].to(device)).detach().to('cpu').numpy()), 
cz_train_ds.target_encoder.inverse_transform(cz_train_ds[11230:11235][1].numpy()))

In [ ]:
sm = torch.nn.modules.activation.LogSoftmax(dim=0)

In [ ]:
sklearn.metrics.precision_score(cz_train_ds[:1000][1].numpy(), cz_train_ds[1000:2000][1].numpy(), average=None)

In [ ]:
#retrieve the run from ML flow

### Load model and do inference

In [ ]:
mlflow.search_runs(experiment_names=[exp1.name])

In [ ]:
# do inference with model through mlflow

In [ ]:
#calculate metrics with model on train, val sets

In [ ]:
# visualise results

### Evaluation

In [ ]:
for clf_name, clf_obj in classifiers_dict.items():
    sklearn.metrics.precision_recall_fscore_support(y_train, y_pred_train[clf_name])

In [ ]:
for clf_name, clf_obj in classifiers_dict.items():
    sklearn.metrics.precision_recall_fscore_support(y_val, y_pred_val[clf_name])

We will go into more details on evaluating the performance of a machine learning model in a separate notebook.

### Loading and storing with ML Flow
One additional feature of usingn ML flow for tracking our experiments, is that the trained model is saved as a part of the 

# Exercises

Once you have worked through the tutorial above, you can test your knowledge by adapting the code in the tutorial to experiment with different options for training the model and comparing results.

### Next steps or potential follow on material

Additional excercises in this tutorial material includes:
- Training a CNN autoencoder on CMIP6 data
- Training a RNN on Jena weather dataset for time series prediction. 


### Data statement
This data used in this notebook is derived from the Koppen-Geiger Climate Cliassfication dataset created by GloH2O.

###     References
- [Climate Zones Dataset](https://www.gloh2o.org/koppen/#:~:text=The%20K%C3%B6ppen%2DGeiger%20climate%20classification%20maps%20are%20high%2Dresolution,Climate%20Sensitivity%20(ECS)%2C%20and%20historical%20warming%20trend)
